In [1]:
import os

# If you manually select physical GPU 1, uncomment this BEFORE importing torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import json
import math
import inspect
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model

ROOT = Path("/home/sece2026-student15/twilight")

MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"
OUTPUT_DIR = ROOT / "resume_qa_32b"

TRAIN_FILE = ROOT / "resume_qa/train_sft_draft.jsonl"
VALIDATION_FILE = ROOT / "resume_qa/validation_sft_draft.jsonl"
PROMPT_FILE = ROOT / "resume_qa_llm/system_prompt.txt"

MAX_LENGTH = 4096
BATCH_SIZE = 1
GRAD_ACCUM = 16
EPOCHS = 2
LEARNING_RATE = 2e-5

assert torch.cuda.is_available(), "CUDA is unavailable."
assert torch.cuda.is_bf16_supported(), "BF16 support is required."

# This notebook setup expects one visible GPU.
assert torch.cuda.device_count() == 1, (
    "Select one assigned GPU before importing torch, then restart the kernel."
)

free, total = torch.cuda.mem_get_info(0)

print("GPU:", torch.cuda.get_device_name(0))
print(f"Free VRAM: {free / 1024**3:.1f} GiB")
print(f"Total VRAM: {total / 1024**3:.1f} GiB")

set_seed(42)


def read_jsonl(path):
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


train_examples = read_jsonl(TRAIN_FILE)
validation_examples = read_jsonl(VALIDATION_FILE)
QA_SYSTEM_PROMPT = PROMPT_FILE.read_text(encoding="utf-8")

assert train_examples and validation_examples

assert not (
    {x["student_id"] for x in train_examples}
    & {x["student_id"] for x in validation_examples}
), "Training and validation contain overlapping students."

print("Training examples:", len(train_examples))
print("Validation examples:", len(validation_examples))

GPU: NVIDIA B200
Free VRAM: 98.5 GiB
Total VRAM: 178.3 GiB
Training examples: 1467
Validation examples: 144


In [2]:
tokenizer_32b = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer_32b.pad_token_id is None:
    tokenizer_32b.pad_token = tokenizer_32b.eos_token

tokenizer_32b.padding_side = "right"


def encode_examples(examples, name):
    encoded = []

    for example in examples:
        messages = [
            {"role": "system", "content": QA_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "resume_context": example["context"],
                        "question": example["question"],
                    },
                    ensure_ascii=False,
                ),
            },
        ]

        prompt_ids = tokenizer_32b.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
        )

        answer = example["answer"].strip()
        if not answer:
            raise ValueError("Found an empty training answer.")

        answer_ids = tokenizer_32b(
            answer,
            add_special_tokens=False,
        )["input_ids"] + [tokenizer_32b.eos_token_id]

        input_ids = prompt_ids + answer_ids

        # Preserve the comparison dataset rather than silently dropping rows.
        if len(input_ids) > MAX_LENGTH:
            raise ValueError(
                f"{name}: example for {example['student_id']} "
                f"has {len(input_ids)} tokens. Review its length first."
            )

        encoded.append({
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": [-100] * len(prompt_ids) + answer_ids,
        })

    print(f"{name}: {len(encoded)} examples")
    return Dataset.from_list(encoded)


train_dataset_32b = encode_examples(train_examples, "Training")
validation_dataset_32b = encode_examples(
    validation_examples, "Validation"
)


def collate_32b(features):
    batch = tokenizer_32b.pad(
        [
            {
                "input_ids": item["input_ids"],
                "attention_mask": item["attention_mask"],
            }
            for item in features
        ],
        padding=True,
        pad_to_multiple_of=8,
        return_tensors="pt",
    )

    labels = torch.full_like(batch["input_ids"], -100)

    for index, item in enumerate(features):
        length = len(item["labels"])
        labels[index, :length] = torch.tensor(
            item["labels"], dtype=torch.long
        )

    batch["labels"] = labels
    return batch


sample = train_dataset_32b[0]
print("\nAnswer target:")
print(tokenizer_32b.decode(
    [token for token in sample["labels"] if token != -100],
    skip_special_tokens=False,
))

Training: 1467 examples
Validation: 144 examples

Answer target:
Developed an application that detects sudden phone shaking using accelerometer sensors to activate SOS mode automatically.<|im_end|>


In [3]:
model_32b = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map={"": 0},  # Entire model on the one selected GPU.
)

model_32b.config.use_cache = False
model_32b.config.pad_token_id = tokenizer_32b.pad_token_id

model_32b = get_peft_model(
    model_32b,
    LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules="all-linear",
        bias="none",
    ),
)

model_32b.print_trainable_parameters()

free, total = torch.cuda.mem_get_info(0)
print(f"\nFree VRAM after loading: {free / 1024**3:.1f} GiB")
print("Ready for 32B QA training.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/17 [00:00<?, ?it/s]

trainable params: 134,217,728 || all params: 32,898,094,080 || trainable%: 0.4080

Free VRAM after loading: 36.8 GiB
Ready for 32B QA training.


In [4]:
updates_per_epoch = math.ceil(
    math.ceil(len(train_dataset_32b) / BATCH_SIZE) / GRAD_ACCUM
)
warmup_steps = max(
    1, math.ceil(updates_per_epoch * EPOCHS * 0.05)
)

settings = dict(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    optim="adamw_torch",

    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    logging_steps=10,
    logging_first_step=True,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    prediction_loss_only=True,
    label_names=["labels"],
    report_to="none",
    dataloader_num_workers=0,
    seed=42,
)

# Account for the evaluation-argument rename across installed versions.
argument_names = inspect.signature(TrainingArguments).parameters
evaluation_key = (
    "eval_strategy"
    if "eval_strategy" in argument_names
    else "evaluation_strategy"
)
settings[evaluation_key] = "epoch"

training_args_32b = TrainingArguments(**settings)

trainer_settings = dict(
    model=model_32b,
    args=training_args_32b,
    train_dataset=train_dataset_32b,
    eval_dataset=validation_dataset_32b,
    data_collator=collate_32b,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=1)
    ],
)

trainer_parameters = inspect.signature(Trainer).parameters
tokenizer_key = (
    "processing_class"
    if "processing_class" in trainer_parameters
    else "tokenizer"
)
trainer_settings[tokenizer_key] = tokenizer_32b

trainer_32b = Trainer(**trainer_settings)

baseline_32b = trainer_32b.evaluate()
print(
    "\n32B validation answer loss BEFORE:",
    baseline_32b["eval_loss"],
)

# Save the baseline result before beginning training.
trainer_32b.save_metrics("baseline", baseline_32b)

training_result_32b = trainer_32b.train()

# The best checkpoint from this training run is now loaded.
final_32b = trainer_32b.evaluate()
final_32b["baseline_eval_loss"] = baseline_32b["eval_loss"]

ADAPTER_32B = OUTPUT_DIR / "adapter"

trainer_32b.save_model(str(ADAPTER_32B))
tokenizer_32b.save_pretrained(str(ADAPTER_32B))

trainer_32b.save_metrics("train", training_result_32b.metrics)
trainer_32b.save_metrics("eval", final_32b)

(OUTPUT_DIR / "system_prompt.txt").write_text(
    QA_SYSTEM_PROMPT,
    encoding="utf-8",
)

print("\nTRAINING COMPLETED")
print(f"Before: {baseline_32b['eval_loss']:.4f}")
print(f"After:  {final_32b['eval_loss']:.4f}")
print("Saved adapter:", ADAPTER_32B)

if final_32b["eval_loss"] >= baseline_32b["eval_loss"]:
    print("The adapter did not improve over the unmodified 32B model.")

The model is already on multiple devices. Skipping the move to device specified in `args`.
You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



32B validation answer loss BEFORE: 3.2087137699127197


Epoch,Training Loss,Validation Loss,Model Preparation Time
1,0.163300,0.167862,0.026800
2,0.150600,0.157964,0.026800



TRAINING COMPLETED
Before: 3.2087
After:  0.1580
Saved adapter: /home/sece2026-student15/twilight/resume_qa_32b/adapter
